# Fine-tuning Mistral 7B with LoRA / QLoRA on Google Colab

**Before you start**
1. *Runtime → Change runtime type → GPU* (pick the largest available, e.g. A100).
2. Add your Hugging Face token in Colab **Secrets** (key icon on the left) as `HF_TOKEN`, and accept the model's terms on its Hugging Face page.
3. Put `train.jsonl`, `val.jsonl` and `test.jsonl` in `MyDrive/mistral_ft/data/` (chat format: `{"messages": [{"role": "user", ...}, {"role": "assistant", ...}]}`).

Checkpoints are saved to Google Drive, so if the session disconnects you can re-run all cells and training resumes.

In [ ]:
!nvidia-smi

In [ ]:
# Pin versions once a run works, so results stay reproducible
!pip install -q -U transformers datasets peft trl bitsandbytes accelerate

In [ ]:
import os
from google.colab import userdata, drive
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))   # token from Colab Secrets, never pasted in code
drive.mount("/content/drive")

## Configuration

In [ ]:
import torch

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
BASE = "/content/drive/MyDrive/mistral_ft"
DATA_DIR = f"{BASE}/data"
RUN_NAME = "distilled_qwen_nothink_v1"   # change for each experiment
OUT_DIR = f"{BASE}/outputs/{RUN_NAME}"

gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
USE_4BIT = gpu_mem_gb < 30              # QLoRA on smaller GPUs, plain LoRA on 40/80 GB A100s
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(torch.cuda.get_device_name(0), f"{gpu_mem_gb:.0f} GB",
      "| mode:", "QLoRA (4-bit)" if USE_4BIT else "LoRA (16-bit)", "| dtype:", DTYPE)

## Load model and tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

quant = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True,
) if USE_4BIT else None

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant, device_map="auto", torch_dtype=DTYPE
)

## Load data and check one example

In [ ]:
from datasets import load_dataset

data = load_dataset("json", data_files={
    "train": f"{DATA_DIR}/train.jsonl",
    "validation": f"{DATA_DIR}/val.jsonl",
    "test": f"{DATA_DIR}/test.jsonl",
})
print(data)
# Always look at what the model will actually see
print(tokenizer.apply_chat_template(data["train"][0]["messages"], tokenize=False))

## Train

In [ ]:
import glob
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

args = SFTConfig(
    output_dir=OUT_DIR,                  # on Drive, survives disconnects
    num_train_epochs=2,
    per_device_train_batch_size=4 if not USE_4BIT else 2,
    gradient_accumulation_steps=4 if not USE_4BIT else 8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_length=1024,
    gradient_checkpointing=True,
    logging_steps=10,
    eval_strategy="steps", eval_steps=50,
    save_strategy="steps", save_steps=50, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    bf16=(DTYPE == torch.bfloat16), fp16=(DTYPE == torch.float16),
    report_to="none",                    # or "wandb" / "mlflow"
    seed=42,
)

trainer = SFTTrainer(
    model=model, args=args,
    train_dataset=data["train"], eval_dataset=data["validation"],
    peft_config=peft_config, processing_class=tokenizer,
)

# Resume automatically if a previous session saved checkpoints
has_ckpt = len(glob.glob(f"{OUT_DIR}/checkpoint-*")) > 0
trainer.train(resume_from_checkpoint=has_ckpt)
trainer.save_model(f"{OUT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{OUT_DIR}/final_adapter")

## Quick check on validation examples (the test set stays untouched until the final evaluation)

In [ ]:
import json

trained = trainer.model
trained.eval()

def generate(messages):
    prompt = [m for m in messages if m["role"] != "assistant"]
    ids = tokenizer.apply_chat_template(prompt, add_generation_prompt=True,
                                        return_tensors="pt").to(trained.device)
    with torch.no_grad():
        out = trained.generate(ids, max_new_tokens=512, do_sample=False)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

n, valid = 20, 0
for ex in data["validation"].select(range(n)):
    answer = generate(ex["messages"])
    try:
        json.loads(answer); valid += 1
    except json.JSONDecodeError:
        print("INVALID:", answer[:200])
print(f"Valid JSON: {valid}/{n}")

## Optional: save the adapter to a private Hugging Face repository

In [ ]:
# trained.push_to_hub("your-username/mistral7b-maintie-extractor", private=True)
# tokenizer.push_to_hub("your-username/mistral7b-maintie-extractor", private=True)